# Building a custom infrastructure metrics dashboard with Elastic Agent Builder

Companion notebook for [the article](https://www.elastic.co/observability-labs/blog/custom-infrastructure-metrics-dashboard-agent-builder).

Requirements:

- Elasticsearch and Kibana 9.5+ with an Enterprise license or trial, Agent Builder and an LLM connector.
- Docker.
- A `.env` copied from `.env.example`. Create the API key as a superuser, in Kibana **Dev Tools**:

```
POST /_security/api_key
{ "name": "custom-infrastructure-metrics-dashboard-agent-builder-notebook", "role_descriptors": {} }
```

`metrics-starlink.otel-default` must not exist yet: only a new time series data stream accepts the history this notebook backfills.

In [ ]:
%pip install -q -r requirements.txt

## 1. Connect

In [ ]:
import json
import os
import pathlib
import subprocess
import sys
import threading
import time
import uuid
import warnings

import requests
from dotenv import load_dotenv
from elasticsearch import Elasticsearch, ElasticsearchWarning

sys.path.insert(0, "poller")
from starlink_poller import (
    ALERTS,
    EXPORT_INTERVAL_S,
    GAUGES,
    SITES,
    SimulatedDish,
)  # noqa: E402

load_dotenv()
# ES|QL adds LIMIT 1000 to queries without one; these return a few rows.
warnings.filterwarnings(
    "ignore", message="No limit defined", category=ElasticsearchWarning
)

SPACE = os.getenv("KIBANA_SPACE", "default")
KIBANA_URL = os.environ["KIBANA_URL"].rstrip("/")
KBN_BASE = KIBANA_URL if SPACE == "default" else f"{KIBANA_URL}/s/{SPACE}"
COLLECTOR_TOKEN = os.environ["COLLECTOR_TOKEN"]
AGENT_ID = os.getenv("AGENT_ID") or "elastic-ai-agent"
CONNECTOR_ID = os.getenv("CONNECTOR_ID") or None

# Everything this notebook creates. Clean up deletes exactly these.
STREAM = "metrics-starlink.otel-default"
COLLECTOR = "http://localhost:14318"  # docker-compose.yml maps 14318 -> 4318
STATE = pathlib.Path("runs/state.json")  # conversation and dashboard ids
STATE.parent.mkdir(exist_ok=True)

es = Elasticsearch(
    os.environ["ELASTICSEARCH_URL"],
    api_key=os.environ["ELASTICSEARCH_API_KEY"],
    request_timeout=120,
)
kibana = requests.Session()
kibana.headers.update(
    {
        "Authorization": f"ApiKey {os.environ['ELASTICSEARCH_API_KEY']}",
        "kbn-xsrf": "true",
        "elastic-api-version": "2023-10-31",
    }
)


def kbn(method, path, ok=(), **kwargs):
    """Call a Kibana API in the configured space. Raises on errors not listed in `ok`."""
    response = kibana.request(method, f"{KBN_BASE}{path}", timeout=120, **kwargs)
    if response.status_code >= 400 and response.status_code not in ok:
        raise RuntimeError(
            f"{method} {path} -> {response.status_code}: {response.text[:1000]}"
        )
    return response.json() if response.content and response.ok else {}


def esql(query, title):
    """Run an ES|QL query and print the result as a text table."""
    print(f"\n{title}\n{es.esql.query(query=query, format='txt').body}")


def save_state(**values):
    state = json.loads(STATE.read_text()) if STATE.exists() else {}
    STATE.write_text(json.dumps({**state, **values}, indent=2))


print(
    f"elasticsearch {es.info()['version']['number']} | space={SPACE} | agent={AGENT_ID}"
)
if (
    es.options(ignore_status=404).indices.get_data_stream(name=STREAM).meta.status
    == 200
):
    raise SystemExit(
        f"{STREAM} already exists: delete it first (DELETE /_data_stream/{STREAM})"
    )

## 2. Send the dish readings as OpenTelemetry metrics

In [ ]:
subprocess.run(["docker", "compose", "up", "-d"], check=True)

AUTH = {"Authorization": f"Bearer {COLLECTOR_TOKEN}"}
for _ in range(60):
    try:
        if requests.post(
            f"{COLLECTOR}/v1/metrics", headers=AUTH, json={}, timeout=5
        ).ok:
            break
    except requests.ConnectionError:
        pass
    time.sleep(2)
else:
    raise SystemExit("the collector did not start: docker compose logs collector")
print("collector accepting OTLP on", COLLECTOR)

The poller's incidents happen 20 to 85 minutes into a run. Instead of waiting, the next cell replays the first 98 minutes with past timestamps through the same collector (the question in section 3 then comes about 40 minutes after the `farm-iowa` rain fade ends, as in the article), and keeps sending live readings until Clean up.

In [ ]:
MINUTES = 98


def attr(key, value):
    return {"key": key, "value": {"stringValue": value}}


def point(ts, value, integer, attributes=()):
    p = {"timeUnixNano": str(int(ts * 1e9)), "attributes": list(attributes)}
    if integer:
        p["asInt"] = str(int(value))  # OTLP/JSON encodes int64 as a string
    else:
        p["asDouble"] = float(value)
    return p


def otlp(site, readings):
    """OTLP/JSON with the poller's resource attributes and metrics for one site."""
    spec = SITES[site]
    metrics = [
        {
            "name": name,
            "unit": unit,
            "description": desc,
            "gauge": {"dataPoints": [point(ts, r[key], integer) for ts, r in readings]},
        }
        for name, unit, key, desc, integer in GAUGES
    ]
    metrics.append(
        {
            "name": "starlink.dish.alert.active",
            "unit": "1",
            "gauge": {
                "dataPoints": [
                    point(ts, r["alerts"][a], True, [attr("alert.name", a)])
                    for ts, r in readings
                    for a in ALERTS
                ]
            },
        }
    )
    resource = [
        attr("service.name", "starlink-poller"),
        attr("host.name", f"gw-{site}"),
        attr("site.name", site),
        attr("site.type", spec["type"]),
        attr("site.region", spec["region"]),
        attr("starlink.dish.hardware", spec["hw"]),
    ]
    scope = {
        "scope": {"name": "starlink-poller", "version": "1.0.0"},
        "metrics": metrics,
    }
    return {
        "resourceMetrics": [
            {"resource": {"attributes": resource}, "scopeMetrics": [scope]}
        ]
    }


def send(site, readings):
    requests.post(
        f"{COLLECTOR}/v1/metrics", headers=AUTH, json=otlp(site, readings), timeout=30
    ).raise_for_status()


now = int(time.time()) // EXPORT_INTERVAL_S * EXPORT_INTERVAL_S
start = now - MINUTES * 60
dishes = {site: SimulatedDish(site, spec, start) for site, spec in SITES.items()}

timestamps = range(start, now + 1, EXPORT_INTERVAL_S)
for site, dish in dishes.items():
    readings = [(ts, dish.sample_at(ts)) for ts in timestamps]
    for i in range(0, len(readings), 60):  # 10 minutes per request
        send(site, readings[i : i + 60])
print(
    f"sent {len(timestamps)} readings per site, ending {time.strftime('%H:%M:%S', time.gmtime(now))} UTC"
)

stop_live = threading.Event()


def live(ts):
    while not stop_live.wait(max(0, ts - time.time())):
        for site, dish in dishes.items():
            try:
                send(site, [(ts, dish.sample_at(ts))])
            except requests.RequestException as exc:
                print("live send failed:", exc)
        ts += EXPORT_INTERVAL_S


threading.Thread(target=live, args=(now + EXPORT_INTERVAL_S,), daemon=True).start()

In [ ]:
for _ in range(30):
    es.options(ignore_status=404).indices.refresh(index=STREAM)
    if es.options(ignore_status=404).count(index=STREAM).get("count", 0) > 0:
        break
    time.sleep(5)

data_stream = es.indices.get_data_stream(name=STREAM)["data_streams"][0]
backing = data_stream["indices"][0]["index_name"]
mode = es.indices.get_settings(index=backing, name="index.mode")[backing]["settings"][
    "index"
]["mode"]
print(f"template: {data_stream['template']} | index.mode: {mode}")

esql(
    f"""
FROM {STREAM}
| STATS docs = COUNT(*), sites = COUNT_DISTINCT(site.name), first_reading = MIN(@timestamp), last_reading = MAX(@timestamp)
""",
    "Documents in the stream",
)

fields = es.indices.get_field_mapping(index=backing, fields="*starlink.dish.*")[
    backing
]["mappings"]
metrics = sorted(
    name
    for name, field in fields.items()
    if any(m.get("time_series_metric") for m in field["mapping"].values())
)
print(f"\n{len(metrics)} metrics:", *metrics, sep="\n")

## 3. Ask the agent about your data

The prompts go through `POST /api/agent_builder/converse/async`, the streaming endpoint the chat UI uses, all in one conversation.

In [ ]:
CONVERSATION_ID = str(uuid.uuid4())
save_state(conversation_id=CONVERSATION_ID)


def converse(prompt_file):
    """Send one prompt from prompts/ and return the finished round."""
    body = {
        "input": pathlib.Path("prompts", prompt_file).read_text().strip(),
        "agent_id": AGENT_ID,
        "conversation_id": CONVERSATION_ID,
        "capabilities": {"visualizations": True},
    }
    if CONNECTOR_ID:
        body["connector_id"] = CONNECTOR_ID
    started, event, done = time.time(), None, None
    with kibana.post(
        f"{KBN_BASE}/api/agent_builder/converse/async",
        json=body,
        stream=True,
        timeout=(10, 900),
    ) as response:
        response.raise_for_status()
        for line in response.iter_lines(decode_unicode=True):
            if line.startswith("event:"):
                event = line[6:].strip()
            elif line.startswith("data:"):
                payload = json.loads(line[5:])
                if event == "error":
                    raise RuntimeError(payload.get("error"))
                if event == "round_complete":
                    done = payload["data"]["round"]
    if done is None:
        raise RuntimeError("the stream ended without round_complete")
    if done.get("status") == "awaiting_prompt":
        raise RuntimeError(
            f"the agent is waiting for input: {done.get('pending_prompts')}"
        )

    tools = [
        step["tool_id"]
        for step in done.get("steps", [])
        if step.get("type") == "tool_call"
    ]
    print(f"{time.time() - started:.0f} s | tools: {', '.join(tools) or 'none'}\n")
    print(done["response"]["message"])
    return done


round_1 = converse("01-whats-in-the-data.md")

In [ ]:
round_2 = converse("02-ask-a-question.md")

In [ ]:
last_fade = es.esql.query(
    query=f"""
FROM {STREAM}
| WHERE site.name == "farm-iowa" AND starlink.dish.pop_ping.drop_rate > 0.02
| STATS last_above_2pct = MAX(@timestamp)
"""
)["values"][0][0]
print(
    f"farm-iowa rain fade, last sample above 2% drop: {last_fade} | question asked: {round_2['started_at']}"
)

## 4. Describe the dashboard you want

In [ ]:
round_3 = converse("03-build-dashboard.md")

In [ ]:
def dashboard_attachment_id(round_):
    """Id of the dashboard attachment that generate_dashboard returned in this round."""
    for step in round_.get("steps", []):
        if step.get("tool_id") != "platform.dashboard.generate_dashboard":
            continue
        for result in step.get("results", []):
            if result.get("type") == "dashboard":
                if result["data"].get("failures"):
                    print("panel failures:", result["data"]["failures"])
                return result["data"]["attachment_id"]
    return None


def latest_dashboard(attachment_id):
    """Current version of the dashboard attachment in the conversation."""
    conversation = kbn("GET", f"/api/agent_builder/conversations/{CONVERSATION_ID}")
    attachment = next(
        a for a in conversation["attachments"] if a["id"] == attachment_id
    )
    current = attachment["current_version"]
    return current, next(
        v["data"] for v in attachment["versions"] if v["version"] == current
    )


def find_esql(obj):
    if isinstance(obj, dict):
        if obj.get("type") == "esql" and isinstance(obj.get("query"), str):
            yield obj["query"]
        for value in obj.values():
            yield from find_esql(value)
    elif isinstance(obj, list):
        for value in obj:
            yield from find_esql(value)


def describe(version, dashboard, show_esql=True):
    print(
        f"{dashboard['title']} (version {version}) | time range: {dashboard.get('time_range')}"
    )
    print(
        "  filters:",
        ", ".join(
            c.get("config", {}).get("title", c.get("type"))
            for c in dashboard.get("pinned_panels", [])
        ),
    )
    for item in dashboard["panels"]:
        if "panels" in item:
            print(f"  section: {item.get('title')}")
        for panel in item.get("panels", [item]):
            config = panel.get("config", {})
            print(
                f"    [{panel['type']}] {config.get('title', '(untitled)')}  grid={panel.get('grid')}"
            )
            if show_esql:
                for query in find_esql(config):
                    print("      " + " ".join(query.split()))


ATTACHMENT_ID = dashboard_attachment_id(round_3)
assert ATTACHMENT_ID, "the agent did not call generate_dashboard"
describe(*latest_dashboard(ATTACHMENT_ID))

## 5. Adjust it the same way you built it

In [ ]:
converse("04-adjust-dashboard.md")
version, dashboard = latest_dashboard(ATTACHMENT_ID)
print()
describe(version, dashboard, show_esql=False)

The UI's **Save** button maps to `POST /api/dashboards`. That API rejects Vega panels: if the agent drew one, save from the preview in Agent Builder instead.

In [ ]:
DASHBOARD_FIELDS = {
    "title",
    "description",
    "panels",
    "pinned_panels",
    "filters",
    "options",
    "query",
    "refresh_interval",
    "tags",
    "time_range",
    "access_control",
}
saved = kbn(
    "POST",
    "/api/dashboards",
    json={k: v for k, v in dashboard.items() if k in DASHBOARD_FIELDS},
)
save_state(dashboard_id=saved["id"])
print(
    f"saved {saved['data']['title']!r}: {KBN_BASE}/app/dashboards#/view/{saved['id']}"
)

## 6. Use the dashboard

In [ ]:
esql(
    f"""
TS {STREAM}
| WHERE @timestamp >= NOW() - 2 hours
| STATS max_alert = MAX(MAX_OVER_TIME(starlink.dish.alert.active)),
        max_obstruction = MAX(MAX_OVER_TIME(starlink.dish.obstruction.fraction)),
        max_drop = MAX(MAX_OVER_TIME(starlink.dish.pop_ping.drop_rate))
  BY site.name, minute = TBUCKET(1 minute)
| EVAL cause = CASE(max_alert == 1, "Dish alert", max_obstruction > 0.01, "Obstruction", max_drop > 0.02, "Weather")
| WHERE cause IS NOT NULL
| STATS bad_minutes = COUNT(*) BY site.name, cause
| SORT site.name, bad_minutes DESC
""",
    "Bad minutes per site by cause, last 2 hours",
)

esql(
    f"""
FROM {STREAM}
| WHERE starlink.dish.alert.active == 1 AND @timestamp >= NOW() - 2 hours
| STATS started = MIN(@timestamp), ended = MAX(@timestamp) BY site.name, alert.name
| SORT started
""",
    "Dish alert episodes",
)

esql(
    f"""
FROM {STREAM}
| WHERE @timestamp >= NOW() - 2 hours
| STATS sites_reporting = COUNT_DISTINCT(site.name), avg_drop_pct = ROUND(AVG(starlink.dish.pop_ping.drop_rate) * 100, 1)
""",
    "Fleet status",
)

## 7. Clean up

Deletes only what the notebook created. The ids come from `runs/state.json`, so this cell also works in a new session.

In [ ]:
if "stop_live" in globals():
    stop_live.set()
state = json.loads(STATE.read_text()) if STATE.exists() else {}

if "dashboard_id" in state:
    kbn("DELETE", f"/api/dashboards/{state['dashboard_id']}", ok=(404,))
if "conversation_id" in state:
    kbn(
        "DELETE",
        f"/api/agent_builder/conversations/{state['conversation_id']}",
        ok=(404,),
    )
subprocess.run(["docker", "compose", "down"], check=True)
es.options(ignore_status=404).indices.delete_data_stream(name=STREAM)
STATE.unlink(missing_ok=True)
print("deleted the dashboard, the conversation and", STREAM)